In [ ]:
pip install transformers datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00


# Chargement du modèle et du dataset

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from datasets import load_dataset

model_name = "JohnOfGod33/distilbert-base-cased-squad-qa"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

dataset = load_dataset("rajpurkar/squad_v2")
validation_data = dataset["validation"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/323 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

squad_v2/train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

squad_v2/validation-00000-of-00001.parqu(…):   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

# Fonction de prédiction

In [ ]:
import torch
import torch.nn.functional as F

def predict(question, context, threshold=0.03):
    inputs = tokenizer(question, context, return_tensors="pt", truncation=True)

    with torch.no_grad():
        outputs = model(**inputs)

    start_logits = outputs.start_logits
    end_logits = outputs.end_logits

    start = torch.argmax(start_logits)
    end = torch.argmax(end_logits) + 1

    no_answer_prob = float(
        F.softmax(start_logits, dim=-1)[0][0] *
        F.softmax(end_logits, dim=-1)[0][0]
    )

    if no_answer_prob > threshold:
        return "", no_answer_prob

    answer = tokenizer.decode(inputs["input_ids"][0][start:end])
    return answer, no_answer_prob

# Évaluation avec métriques SQuAD v2

In [ ]:
import evaluate

metric = evaluate.load("squad_v2")

# Boucle d’évaluation

In [ ]:
from tqdm import tqdm

predictions = []
references = []

for example in tqdm(validation_data):
    question = example["question"]
    context = example["context"]

    pred_answer, no_answer_prob = predict(question, context)

    predictions.append({
        "id": example["id"],
        "prediction_text": pred_answer,
        "no_answer_probability": no_answer_prob
    })

    references.append({
        "id": example["id"],
        "answers": example["answers"]
    })

results = metric.compute(predictions=predictions, references=references)

print(results)

100%|██████████| 11873/11873 [1:18:21<00:00,  2.53it/s]


{'exact': 60.11117661922008, 'f1': 63.1694376166282, 'total': 11873, 'HasAns_exact': 45.54655870445344, 'HasAns_f1': 51.671851015895186, 'HasAns_total': 5928, 'NoAns_exact': 74.63414634146342, 'NoAns_f1': 74.63414634146342, 'NoAns_total': 5945, 'best_exact': 60.15328897498526, 'best_exact_thresh': 0.029465842992067337, 'best_f1': 63.20366358497707, 'best_f1_thresh': 0.029465842992067337}


-Exact Match (EM) = réponses exactes

-F1 Score = qualité globale

-HasAns = performance quand réponse existe

-NoAns = capacité à dire "pas de réponse"